# **MCP Client**

MCP is a standardised way to integrate tools with your applications. As defined by Anthropic:
> **MCP is an open protocol that standardizes how your LLM Applicaitons connect to and work with your tools and data sources.**

Problem with the Tools:
- N*M Client-side Integrations
- More Maintanance
- Time Consuming

With MCP:
- Server do the heavy lifting
- Client simply plugin and play

### **Connecting your MCP Client with Other MCP Servers**

Check out:
- https://github.com/punkpeye/awesome-mcp-servers
- https://glama.ai/mcp/servers
- https://mcp.so/servers

In [1]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model="gpt-4o-mini",
                 temperature=0.0)

In [2]:
# !pip install langchain-mcp-adapters

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

Each server must include 'transport' with one of: 'stdio', 'sse', 'websocket', 'http'.

In [4]:
# ! pip install uv

In [5]:
client = MultiServerMCPClient(
    {
        "time": {
          "transport": "stdio",
          "command": "uvx",
          "args": [
            "mcp-server-time",
            "--local-timezone=America/New_York"
          ]
        }
    }
)

In [6]:
time_toolset = await client.get_tools()

time_toolset

[StructuredTool(name='get_current_time', description='Get current time in a specific timezones', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x116988860>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no source timezone provided by the user."}, 'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'}, 'target_timezone': {'type': 'string', 'description': "Target IANA timezone 

In [7]:
print(f"Loaded {len(time_toolset)} MCP Tools: {[tool.name for tool in time_toolset]}")

Loaded 2 MCP Tools: ['get_current_time', 'convert_time']


In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=time_toolset
)

In [9]:
response = await agent.ainvoke({"messages": "What time is it?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What time is it?
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_FeYSFLezw3GfYbxPRRv2ybBd)
 Call ID: call_FeYSFLezw3GfYbxPRRv2ybBd
  Args:
    timezone: America/New_York
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "America/New_York",\n  "datetime": "2026-03-07T08:23:04-05:00",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_7617f488-48a4-41d4-8399-55bc5e3abfdd'}]
================================== Ai Message ==================================

The current time in New York is 8:23 AM on Saturday, March 7, 2026.


In [10]:
response = await agent.ainvoke({"messages": "What time is it in India?"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What time is it in India?
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_rkCbwO6qnDUP71O9cV7oEZG6)
 Call ID: call_rkCbwO6qnDUP71O9cV7oEZG6
  Args:
    timezone: Asia/Kolkata
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Kolkata",\n  "datetime": "2026-03-07T18:53:09+05:30",\n  "day_of_week": "Saturday",\n  "is_dst": false\n}', 'id': 'lc_fd23a415-c8d5-4f2e-a69a-365a7067a3c2'}]
================================== Ai Message ==================================

The current time in India (Asia/Kolkata) is 6:53 PM on Saturday, March 7, 2026.


## **Building Flight Agent**

In [11]:
from langchain_mcp_adapters.client import MultiServerMCPClient

Each server must include 'transport' with one of: 'stdio', 'sse', 'websocket', 'http'.

In [12]:
client = MultiServerMCPClient(
    {
        "my_travel_server": {
          "transport": "streamable_http",
          "url": "https://mcp.kiwi.com"
        }
    }
)

flight_toolset = await client.get_tools()

print(f"Loaded {len(flight_toolset)} MCP Tools: {[tool.name for tool in flight_toolset]}")

Loaded 2 MCP Tools: ['search-flight', 'feedback-to-devs']


In [13]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                 model="gpt-4o-mini",
                 temperature=0.0)

In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=time_toolset + flight_toolset,
    system_prompt="You are a travel agent. Dont ask any follow-up questions."
)

In [20]:
response = await agent.ainvoke({"messages": "Get me a direct flight details from nyc to new delhi five days from today."})

for msg in response["messages"]:
    msg.pretty_print()

McpError: MCP error -32602: MCP error -32602: Invalid arguments for tool search-flight: [
  {
    "code": "custom",
    "message": "Dates must be in the future. Current date is 07/03/2026",
    "path": [
      "departureDate"
    ]
  }
]